# Notebook 08 — Climate and Drought Analysis
**Stage 8 of the Ghana Groundwater–Subsidence Pipeline**

Analyses climate variability and drought stress across coastal Ghana using:
- **CHIRPS v2.0** monthly precipitation (1981–present, already downloaded)
- **TerraClimate PDSI** Palmer Drought Severity Index (2002–2025)
- **TerraClimate PET** potential evapotranspiration (2002–2025)

Links climate signals to GWSA (from Notebook 03/04) and feeds drought indicators into the UGPI (Notebook 09) and CGSRI (Notebook 10).

**Outputs**
- `data/processed/climate_indices/rainfall_anomaly_monthly.csv`
- `data/processed/climate_indices/pdsi_annual_mean.tif`
- `data/processed/climate_indices/drought_frequency.tif`
- `data/processed/climate_indices/terraclimate_coastal_timeseries.csv`
- `outputs/figures/chirps_rainfall_timeseries.png`
- `outputs/figures/pdsi_timeseries.png`
- `outputs/figures/drought_frequency_map.png`
- `outputs/figures/gwsa_vs_pdsi_comparison.png`
- `outputs/tables/drought_episodes_pdsi.csv`

In [ ]:
import os, sys
import pathlib; os.chdir(str(pathlib.Path.cwd().parent) if pathlib.Path.cwd().name == 'notebooks' else str(pathlib.Path.cwd()))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from pathlib import Path
import pymannkendall as mk
from scipy import stats
from scipy.ndimage import uniform_filter

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT         = Path('.')
TC_DIR       = ROOT / 'data' / 'raw' / 'terraclimate'
CHIRPS_FILE  = ROOT / 'data' / 'raw' / 'chirps' / 'chirps-v2.0.monthly.nc'
OUT_CLI      = ROOT / 'data' / 'processed' / 'climate_indices'
OUT_GW       = ROOT / 'data' / 'processed' / 'groundwater_anomalies'
BOUND_FILE   = ROOT / 'data' / 'raw' / 'boundaries' / 'study_area_coastal_50km.gpkg'
FIG_DIR      = ROOT / 'outputs' / 'figures'
TAB_DIR      = ROOT / 'outputs' / 'tables'
for d in [OUT_CLI, FIG_DIR, TAB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

GHANA_LAT = slice(4.0, 11.5)
GHANA_LON = slice(-3.5, 1.5)

GRACE_START = '2002-01-01'
GRACE_END   = '2025-12-31'

PDSI_BINS   = [-np.inf, -4.0, -3.0, -2.0, -1.0, 1.0, 2.0, 3.0, np.inf]
PDSI_LABELS = ['Extreme drought','Severe drought','Moderate drought',
               'Mild drought','Near normal','Mild wet','Moderate wet','Extreme wet']
PDSI_COLORS = ['#8B0000','#CC0000','#FF4500','#FF8C00',
               '#90EE90','#4682B4','#1E90FF','#000080']

print('Imports OK')
print(f'CHIRPS file exists: {CHIRPS_FILE.exists()}')
tc_pdsi_files = sorted(TC_DIR.glob('TerraClimate_PDSI_*.nc'))
tc_pet_files  = sorted(TC_DIR.glob('TerraClimate_pet_*.nc'))
print(f'PDSI files found  : {len(tc_pdsi_files)}')
print(f'PET  files found  : {len(tc_pet_files)}')

## 1. CHIRPS Rainfall Analysis

In [ ]:
# ── 1a. Load CHIRPS and clip to Ghana ─────────────────────────────────────
print('Loading CHIRPS...')
chirps_raw = xr.open_dataset(CHIRPS_FILE, engine='netcdf4', chunks={'time': 120})

# Identify the precip variable
pvar = [v for v in chirps_raw.data_vars if 'precip' in v.lower() or 'pr' in v.lower()][0]
chirps_raw = chirps_raw.rename({pvar: 'precip'})

# Rename lat/lon if needed
rename_map = {}
for c in chirps_raw.coords:
    if c.lower() in ('latitude','lat') and c != 'lat':
        rename_map[c] = 'lat'
    if c.lower() in ('longitude','lon') and c != 'lon':
        rename_map[c] = 'lon'
if rename_map:
    chirps_raw = chirps_raw.rename(rename_map)

# Clip
chirps = chirps_raw.sel(lat=GHANA_LAT, lon=GHANA_LON)
chirps = chirps.sel(time=slice(GRACE_START, GRACE_END))
print(f'CHIRPS shape: {chirps["precip"].shape}  time: {chirps.time.values[0]} – {chirps.time.values[-1]}')

In [ ]:
# ── 1b. Spatial mean monthly time series ──────────────────────────────────
chirps_mean = chirps['precip'].mean(dim=['lat','lon']).compute()
chirps_ts = pd.Series(
    chirps_mean.values,
    index=pd.to_datetime(chirps.time.values),
    name='precip_mm'
)

# Monthly climatology (2002-2025 as baseline here since full 91-20 not all loaded)
chirps_clim = chirps_ts.groupby(chirps_ts.index.month).mean()

# Anomaly
chirps_anom = chirps_ts - chirps_ts.index.map(lambda d: chirps_clim[d.month])

# Annual mean
chirps_annual = chirps_ts.resample('YE').sum()
chirps_anom_annual = chirps_anom.resample('YE').sum()

print(f'Mean annual rainfall (coastal Ghana): {chirps_annual.mean():.0f} mm')
print(f'Driest year : {chirps_annual.idxmin().year} ({chirps_annual.min():.0f} mm)')
print(f'Wettest year: {chirps_annual.idxmax().year} ({chirps_annual.max():.0f} mm)')

# Export
df_chirps = pd.DataFrame({
    'date': chirps_ts.index,
    'precip_mm': chirps_ts.values,
    'precip_anom_mm': chirps_anom.values,
})
df_chirps.to_csv(OUT_CLI / 'rainfall_anomaly_monthly.csv', index=False)
print('Saved rainfall_anomaly_monthly.csv')

In [ ]:
# ── 1c. Seasonal cycle ────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 10))
fig.suptitle('CHIRPS Rainfall — Coastal Ghana (2002–2025)', fontsize=13, fontweight='bold')

# Panel 1: annual totals bar chart
ax = axes[0]
colors_bar = ['#CC4444' if v < 0 else '#4477AA' for v in chirps_anom_annual.values]
ax.bar(chirps_anom_annual.index.year, chirps_anom_annual.values, color=colors_bar, width=0.7)
ax.axhline(0, color='k', linewidth=0.8)
ax.set_ylabel('Annual rainfall anomaly (mm)')
ax.set_xlabel('')
ax.set_title('(a) Annual rainfall anomaly relative to 2002–2025 mean', fontsize=10)
ax.tick_params(axis='x', rotation=45)

# Panel 2: monthly time series with 12-month rolling mean
ax = axes[1]
ax.plot(chirps_ts.index, chirps_ts.values, color='steelblue', linewidth=0.6, alpha=0.6, label='Monthly')
ax.plot(chirps_ts.index, chirps_ts.rolling(12, center=True).mean(), color='navy', linewidth=1.5, label='12-month mean')
ax.set_ylabel('Precipitation (mm month⁻¹)')
ax.set_title('(b) Monthly precipitation with 12-month running mean', fontsize=10)
ax.legend(fontsize=8)

# Panel 3: mean seasonal cycle
ax = axes[2]
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
ax.bar(range(1,13), [chirps_clim[m] for m in range(1,13)], color='steelblue', width=0.7)
ax.set_xticks(range(1,13))
ax.set_xticklabels(month_names)
ax.set_ylabel('Mean precipitation (mm month⁻¹)')
ax.set_title('(c) Mean seasonal cycle (2002–2025)', fontsize=10)
ax.text(0.98, 0.95, 'Bimodal rainfall: Mar–Jul peak, Aug–Nov secondary peak',
        transform=ax.transAxes, ha='right', va='top', fontsize=8, color='navy')

plt.tight_layout()
fig.savefig(FIG_DIR / 'chirps_rainfall_timeseries.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved chirps_rainfall_timeseries.png')

## 2. TerraClimate PDSI Analysis

In [ ]:
# ── 2a. Load TerraClimate PDSI files ──────────────────────────────────────
tc_pdsi_files = sorted(TC_DIR.glob('TerraClimate_PDSI_*.nc'))
print(f'Loading {len(tc_pdsi_files)} PDSI files...')

if len(tc_pdsi_files) == 0:
    raise FileNotFoundError(
        'No TerraClimate PDSI files found in data/raw/terraclimate/. '
        'Run: python scripts/download_chirps.py --terraclimate --tc-vars PDSI pet '
        '--start-year 2002 --end-year 2025'
    )

# TerraClimate has descending latitude — use reversed slice or sortby
TC_LAT = slice(11.5, 4.0)   # reversed: descending lat needs high→low
TC_LON = slice(-3.5, 1.5)   # ascending lon works normally

ds_list = []
for f in tc_pdsi_files:
    try:
        ds = xr.open_dataset(f, engine='netcdf4')
        # Rename lat/lon dims if needed
        rename_map = {}
        for c in ds.coords:
            if c.lower() in ('latitude',) and c != 'lat': rename_map[c] = 'lat'
            if c.lower() in ('longitude',) and c != 'lon': rename_map[c] = 'lon'
        if rename_map:
            ds = ds.rename(rename_map)
        ds = ds.sel(lat=TC_LAT, lon=TC_LON)
        ds_list.append(ds)
    except Exception as e:
        print(f'  WARN: could not open {f.name}: {e}')

pdsi_raw = xr.concat(ds_list, dim='time').sortby('time')

# Identify PDSI variable
pdsi_var = [v for v in pdsi_raw.data_vars if 'pdsi' in v.lower() or 'PDSI' in v][0]
pdsi = pdsi_raw[pdsi_var]

# TerraClimate PDSI is stored as integer ×100 — divide to get true PDSI
pmax = float(pdsi.max().values)
if abs(pmax) > 50:
    pdsi = pdsi / 100.0
    print(f'Scaled PDSI by /100 (raw max was {pmax:.1f})')

print(f'PDSI shape : {pdsi.shape}')
print(f'PDSI range : {float(pdsi.min().values):.2f} to {float(pdsi.max().values):.2f}')
print(f'Time range : {pd.Timestamp(pdsi.time.values[0])} – {pd.Timestamp(pdsi.time.values[-1])}')

In [ ]:
# ── 2b. Spatial mean PDSI time series ─────────────────────────────────────
pdsi_spatial_mean = pdsi.mean(dim=['lat','lon']).values
pdsi_time = pd.to_datetime(pdsi.time.values)
pdsi_ts = pd.Series(pdsi_spatial_mean, index=pdsi_time, name='pdsi')

# Annual mean
pdsi_annual = pdsi_ts.resample('YE').mean()

# Drought classification: months with PDSI < -1.0
drought_months = pdsi_ts[pdsi_ts < -1.0]
drought_frac   = (pdsi_ts < -1.0).sum() / len(pdsi_ts) * 100
severe_frac    = (pdsi_ts < -2.0).sum() / len(pdsi_ts) * 100
print(f'PDSI drought months (< -1.0): {len(drought_months)} of {len(pdsi_ts)}  ({drought_frac:.1f}%)')
print(f'PDSI severe drought (< -2.0): {(pdsi_ts < -2.0).sum()} months ({severe_frac:.1f}%)')
print(f'Most negative PDSI: {pdsi_ts.min():.2f} at {pdsi_ts.idxmin().strftime("%Y-%m")}')

In [ ]:
# ── 2c. PDSI drought episodes (consecutive months PDSI < -1.0) ─────────────
records = []
in_drought = False
start_dt = None
min_pdsi = 0.0

for dt, val in pdsi_ts.items():
    if val < -1.0:
        if not in_drought:
            in_drought = True
            start_dt = dt
            min_pdsi = val
        else:
            min_pdsi = min(min_pdsi, val)
    else:
        if in_drought:
            duration = (dt.year - start_dt.year) * 12 + (dt.month - start_dt.month)
            if duration >= 2:  # at least 2 months
                records.append({
                    'start': start_dt.strftime('%Y-%m'),
                    'end':   (dt - pd.DateOffset(months=1)).strftime('%Y-%m'),
                    'duration_months': duration,
                    'min_pdsi': round(min_pdsi, 2),
                    'severity': 'Severe' if min_pdsi < -2.0 else 'Moderate'
                })
            in_drought = False

df_episodes = pd.DataFrame(records).sort_values('min_pdsi')
print(f'PDSI drought episodes (>= 2 months, PDSI < -1.0):')
print(df_episodes.to_string(index=False))
df_episodes.to_csv(TAB_DIR / 'drought_episodes_pdsi.csv', index=False)
print('\nSaved drought_episodes_pdsi.csv')

In [ ]:
# ── 2d. PDSI time series figure ───────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
fig.suptitle('TerraClimate PDSI — Coastal Ghana (2002–2025)', fontsize=13, fontweight='bold')

# Panel 1: monthly PDSI with drought shading
ax = axes[0]
ax.axhline(0, color='k', linewidth=0.7, linestyle='--')
ax.axhline(-1.0, color='orange', linewidth=0.8, linestyle=':', label='Mild drought threshold')
ax.axhline(-2.0, color='red',    linewidth=0.8, linestyle=':', label='Moderate drought threshold')
ax.fill_between(pdsi_ts.index, pdsi_ts.values, 0,
                where=(pdsi_ts.values < 0), color='#FF6B35', alpha=0.35, label='Negative PDSI')
ax.fill_between(pdsi_ts.index, pdsi_ts.values, 0,
                where=(pdsi_ts.values >= 0), color='#4682B4', alpha=0.35, label='Positive PDSI')
ax.plot(pdsi_ts.index, pdsi_ts.rolling(6, center=True).mean(), color='darkred', linewidth=1.5,
        label='6-month mean')
ax.set_ylabel('PDSI')
ax.set_title('(a) Monthly PDSI with 6-month running mean', fontsize=10)
ax.legend(fontsize=7, ncol=2, loc='upper right')
ax.set_ylim(-6, 6)

# Panel 2: annual mean PDSI bar chart
ax = axes[1]
bar_colors = ['#CC4444' if v < 0 else '#4477AA' for v in pdsi_annual.values]
ax.bar(pdsi_annual.index.year, pdsi_annual.values, color=bar_colors, width=0.7, edgecolor='white')
ax.axhline(0,   color='k',      linewidth=0.8)
ax.axhline(-1.0, color='orange', linewidth=0.8, linestyle='--', alpha=0.7)
ax.axhline(-2.0, color='red',    linewidth=0.8, linestyle='--', alpha=0.7)
ax.set_ylabel('Annual mean PDSI')
ax.set_title('(b) Annual mean PDSI', fontsize=10)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
fig.savefig(FIG_DIR / 'pdsi_timeseries.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved pdsi_timeseries.png')

## 3. Drought Frequency Mapping (TerraClimate PDSI)

In [ ]:
# ── 3a. Compute pixel-wise drought frequency ──────────────────────────────
pdsi_np = pdsi.values   # shape: (time, lat, lon)

drought_freq = np.mean(pdsi_np < -1.0, axis=0) * 100    # % months drought
severe_freq  = np.mean(pdsi_np < -2.0, axis=0) * 100
annual_mean  = np.nanmean(pdsi_np, axis=0)

lats = pdsi.lat.values
lons = pdsi.lon.values

# For rioxarray export, sort to ascending lat
pdsi_sorted = pdsi.sortby('lat')
drought_freq_sorted = np.mean(pdsi_sorted.values < -1.0, axis=0) * 100
annual_mean_sorted  = np.nanmean(pdsi_sorted.values, axis=0)
lats_sorted = pdsi_sorted.lat.values
lons_sorted = pdsi_sorted.lon.values

da_drought_freq = xr.DataArray(
    drought_freq_sorted,
    coords={'lat': lats_sorted, 'lon': lons_sorted},
    dims=['lat','lon'],
    attrs={'units': '%', 'long_name': 'Drought frequency PDSI<-1.0 (% months 2002-2025)',
           'crs': 'EPSG:4326'}
)
da_annual_mean = xr.DataArray(
    annual_mean_sorted,
    coords={'lat': lats_sorted, 'lon': lons_sorted},
    dims=['lat','lon'],
    attrs={'units': 'dimensionless', 'long_name': 'Mean PDSI 2002-2025',
           'crs': 'EPSG:4326'}
)

da_drought_freq.rio.set_spatial_dims(x_dim='lon', y_dim='lat', inplace=True)
da_drought_freq.rio.write_crs('EPSG:4326', inplace=True)
da_drought_freq.rio.to_raster(str(OUT_CLI / 'drought_frequency.tif'), driver='GTiff')

da_annual_mean.rio.set_spatial_dims(x_dim='lon', y_dim='lat', inplace=True)
da_annual_mean.rio.write_crs('EPSG:4326', inplace=True)
da_annual_mean.rio.to_raster(str(OUT_CLI / 'pdsi_annual_mean.tif'), driver='GTiff')

print(f'Drought frequency range: {drought_freq.min():.1f}% – {drought_freq.max():.1f}%')
print(f'Mean PDSI range: {annual_mean.min():.2f} – {annual_mean.max():.2f}')
print('Saved drought_frequency.tif and pdsi_annual_mean.tif')

In [ ]:
# ── 3b. Drought frequency map ─────────────────────────────────────────────
# Load study boundary for overlay
try:
    study_area = gpd.read_file(BOUND_FILE)
    has_boundary = True
except Exception:
    has_boundary = False
    print('WARN: study boundary not found — plotting without boundary overlay')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('TerraClimate PDSI — Drought Frequency and Mean Index\nCoastal Ghana 2002–2025',
             fontsize=12, fontweight='bold')

# Panel 1: drought frequency
ax = axes[0]
im = ax.pcolormesh(
    lons, lats, drought_freq,
    cmap='YlOrRd', vmin=0, vmax=50, shading='auto'
)
if has_boundary:
    study_area.boundary.plot(ax=ax, edgecolor='black', linewidth=0.8)
plt.colorbar(im, ax=ax, label='Months in drought (% of 2002–2025)')
ax.set_title('(a) Drought frequency (PDSI < −1.0)', fontsize=10)
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_xlim(-3.5, 1.5)
ax.set_ylim(4.0, 7.5)

# Panel 2: mean PDSI
ax = axes[1]
im2 = ax.pcolormesh(
    lons, lats, annual_mean,
    cmap='RdBu', vmin=-3, vmax=3, shading='auto'
)
if has_boundary:
    study_area.boundary.plot(ax=ax, edgecolor='black', linewidth=0.8)
plt.colorbar(im2, ax=ax, label='Mean PDSI 2002–2025')
ax.set_title('(b) Mean PDSI (negative = drier tendency)', fontsize=10)
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_xlim(-3.5, 1.5)
ax.set_ylim(4.0, 7.5)

plt.tight_layout()
fig.savefig(FIG_DIR / 'drought_frequency_map.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved drought_frequency_map.png')

## 4. TerraClimate PET Analysis

In [ ]:
# ── 4. Load TerraClimate PET files ────────────────────────────────────────
tc_pet_files = sorted(TC_DIR.glob('TerraClimate_pet_*.nc'))
print(f'Loading {len(tc_pet_files)} PET files...')

if len(tc_pet_files) == 0:
    print('NOTE: No PET files found — skipping PET analysis.')
    pet_ts = None
else:
    pet_list = []
    for f in tc_pet_files:
        try:
            ds = xr.open_dataset(f, engine='netcdf4')
            rename_map = {}
            for c in ds.coords:
                if c.lower() in ('latitude',) and c != 'lat': rename_map[c] = 'lat'
                if c.lower() in ('longitude',) and c != 'lon': rename_map[c] = 'lon'
            if rename_map:
                ds = ds.rename(rename_map)
            # TerraClimate has descending latitude
            ds = ds.sel(lat=slice(11.5, 4.0), lon=slice(-3.5, 1.5))
            pet_list.append(ds)
        except Exception as e:
            print(f'  WARN: {f.name}: {e}')

    if pet_list:
        pet_raw = xr.concat(pet_list, dim='time').sortby('time')
        pet_var = [v for v in pet_raw.data_vars if 'pet' in v.lower()][0]
        pet = pet_raw[pet_var]
        # TerraClimate PET is in mm/month ×10 — multiply by 0.1
        pmax_pet = float(pet.max().values)
        if pmax_pet > 5000:
            pet = pet * 0.1
            print(f'Scaled PET by 0.1 (raw max was {pmax_pet:.0f})')
        pet_spatial = pet.mean(dim=['lat','lon']).values
        pet_ts = pd.Series(pet_spatial, index=pd.to_datetime(pet.time.values), name='pet_mm')
        print(f'PET range: {pet_ts.min():.1f} – {pet_ts.max():.1f} mm/month')
    else:
        pet_ts = None
        print('All PET files failed — skipping.')

## 5. Climate–Groundwater Comparison

In [ ]:
# ── 5a. Load GWSA from Notebook 03 ────────────────────────────────────────
gwsa_file = OUT_GW / 'gwsa_monthly.csv'
if not gwsa_file.exists():
    gwsa_file = ROOT / 'data' / 'interim' / 'time_series' / 'gwsa_monthly.csv'

df_gwsa_raw = pd.read_csv(gwsa_file)
time_col = 'date' if 'date' in df_gwsa_raw.columns else 'time'
df_gwsa_raw[time_col] = pd.to_datetime(df_gwsa_raw[time_col])
gwsa_ts = df_gwsa_raw.set_index(time_col)['gwsa_mm']

# Align to PDSI period
common_start = max(gwsa_ts.index.min(), pdsi_ts.index.min())
common_end   = min(gwsa_ts.index.max(), pdsi_ts.index.max())
gwsa_aligned = gwsa_ts.loc[common_start:common_end]
pdsi_aligned = pdsi_ts.reindex(gwsa_aligned.index, method='nearest', tolerance='31D')
chirps_aligned = chirps_ts.reindex(gwsa_aligned.index, method='nearest', tolerance='31D')

print(f'Common period: {common_start.strftime("%Y-%m")} – {common_end.strftime("%Y-%m")}')
print(f'GWSA N={len(gwsa_aligned)}, PDSI N={pdsi_aligned.dropna().shape[0]}')

# Correlation
r_gw_pdsi, p_gw_pdsi, r_gw_rain, p_gw_rain = None, None, None, None
mask = ~(np.isnan(gwsa_aligned.values) | np.isnan(pdsi_aligned.values))
if mask.sum() > 10:
    r_gw_pdsi, p_gw_pdsi = stats.pearsonr(gwsa_aligned.values[mask], pdsi_aligned.values[mask])
    print(f'Pearson r (GWSA vs PDSI): {r_gw_pdsi:.3f}  p={p_gw_pdsi:.4f}')
mask2 = ~(np.isnan(gwsa_aligned.values) | np.isnan(chirps_aligned.values))
if mask2.sum() > 10:
    r_gw_rain, p_gw_rain = stats.pearsonr(gwsa_aligned.values[mask2], chirps_aligned.values[mask2])
    print(f'Pearson r (GWSA vs rainfall): {r_gw_rain:.3f}  p={p_gw_rain:.4f}')

In [ ]:
# ── 5b. Comparison figure ─────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
fig.suptitle('Climate–Groundwater Linkage — Coastal Ghana (2002–2025)',
             fontsize=13, fontweight='bold')

# Panel 1: GWSA
ax = axes[0]
ax.fill_between(gwsa_aligned.index, gwsa_aligned.values, 0,
                where=(gwsa_aligned.values < 0), color='#CC4444', alpha=0.4, label='Deficit')
ax.fill_between(gwsa_aligned.index, gwsa_aligned.values, 0,
                where=(gwsa_aligned.values >= 0), color='#4477AA', alpha=0.4, label='Surplus')
ax.plot(gwsa_aligned.index, gwsa_aligned.values, color='navy', linewidth=1.2)
ax.axhline(0, color='k', linewidth=0.7, linestyle='--')
ax.set_ylabel('GWSA (mm)')
ax.set_title('(a) Groundwater storage anomaly (GRACE–GLDAS)', fontsize=10)
ax.legend(fontsize=8)

# Panel 2: PDSI
ax = axes[1]
ax.fill_between(pdsi_aligned.index, pdsi_aligned.values, 0,
                where=(pdsi_aligned.values < 0), color='#FF6B35', alpha=0.45, label='Drought')
ax.fill_between(pdsi_aligned.index, pdsi_aligned.values, 0,
                where=(pdsi_aligned.values >= 0), color='#4682B4', alpha=0.45, label='Wet')
ax.plot(pdsi_aligned.index, pdsi_aligned.rolling(3, center=True).mean(), color='darkred', linewidth=1.2)
ax.axhline(0, color='k', linewidth=0.7, linestyle='--')
ax.axhline(-1.0, color='orange', linewidth=0.7, linestyle=':')
ax.set_ylabel('PDSI')
ax.set_ylim(-6, 6)
ax.set_title('(b) Palmer Drought Severity Index (TerraClimate)', fontsize=10)
ax.legend(fontsize=8)

# Panel 3: CHIRPS rainfall
ax = axes[2]
ax.bar(chirps_aligned.index, chirps_aligned.values, width=25, color='steelblue', alpha=0.6, label='Rainfall')
ax.plot(chirps_aligned.index, chirps_aligned.rolling(12, center=True).mean(),
        color='navy', linewidth=1.5, label='12-month mean')
ax.set_ylabel('Precipitation (mm month⁻¹)')
ax.set_xlabel('Year')
ax.set_title('(c) CHIRPS monthly precipitation', fontsize=10)
ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(FIG_DIR / 'gwsa_vs_pdsi_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved gwsa_vs_pdsi_comparison.png')

## 6. Compound Climate–Groundwater Drought Index

In [ ]:
# ── 6. Build and export combined climate time series ──────────────────────
df_combined = pd.DataFrame({
    'date':          gwsa_aligned.index,
    'gwsa_mm':       gwsa_aligned.values,
    'pdsi':          pdsi_aligned.reindex(gwsa_aligned.index).values,
    'precip_mm':     chirps_aligned.reindex(gwsa_aligned.index).values,
})
if pet_ts is not None:
    df_combined['pet_mm'] = pet_ts.reindex(gwsa_aligned.index, method='nearest', tolerance='31D').values

# Standardise
for col in ['gwsa_mm','pdsi','precip_mm']:
    mu  = df_combined[col].mean()
    sig = df_combined[col].std()
    df_combined[f'z_{col}'] = (df_combined[col] - mu) / sig

# Compound drought: both PDSI < -1 AND rainfall anomaly < 0
chirps_anom_aligned = chirps_anom.reindex(gwsa_aligned.index, method='nearest', tolerance='31D')
df_combined['compound_drought'] = (
    (df_combined['pdsi'] < -1.0) & (chirps_anom_aligned.values < 0)
).astype(int)

comp_pct = df_combined['compound_drought'].mean() * 100
print(f'Compound drought months: {df_combined["compound_drought"].sum()} ({comp_pct:.1f}% of record)')

df_combined.to_csv(OUT_CLI / 'terraclimate_coastal_timeseries.csv', index=False)
print('Saved terraclimate_coastal_timeseries.csv')

In [ ]:
# ── 6b. Compound drought figure ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
ax.set_title('Compound Climate–Groundwater Drought — Coastal Ghana (2002–2025)',
             fontsize=11, fontweight='bold')

# Shade compound drought months first (background)
for i, row in df_combined.iterrows():
    if row['compound_drought'] == 1:
        ax.axvspan(row['date'] - pd.Timedelta(days=15),
                   row['date'] + pd.Timedelta(days=15),
                   alpha=0.22, color='darkred', zorder=0)

# Lines
line_gw,   = ax.plot(df_combined['date'], df_combined['z_gwsa_mm'],
                     color='navy',     lw=1.5,  label='Standardised GWSA')
line_pdsi, = ax.plot(df_combined['date'], df_combined['z_pdsi'],
                     color='#CC4444', lw=1.2, ls='--', label='Standardised PDSI')
line_rain, = ax.plot(df_combined['date'], df_combined['z_precip_mm'],
                     color='steelblue', lw=1.0, ls=':', label='Standardised rainfall')
ax.axhline(0, color='k', lw=0.7)
ax.set_ylabel('Standardised anomaly (z-score)')
ax.set_xlabel('Year')

# Single combined legend — lines + shading patch
shade_patch = mpatches.Patch(
    color='darkred', alpha=0.4,
    label=f'Compound drought: PDSI < −1 AND rainfall < mean ({comp_pct:.0f}% of months)'
)
ax.legend(handles=[line_gw, line_pdsi, line_rain, shade_patch],
          fontsize=8, loc='upper left', framealpha=0.9)

plt.tight_layout()
fig.savefig(FIG_DIR / 'compound_drought_timeseries.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved compound_drought_timeseries.png')

## 7. Summary Statistics Export

In [ ]:
# ── 7. Summary table ──────────────────────────────────────────────────────
summary = {
    'indicator': [
        'Mean annual rainfall (mm)',
        'Driest year',
        'Wettest year',
        'Mean PDSI (2002-2025)',
        'PDSI drought months (< -1.0, %)',
        'PDSI severe drought months (< -2.0, %)',
        'Compound drought months (%)',
        'Pearson r (GWSA vs PDSI)',
        'Pearson r (GWSA vs rainfall)',
    ],
    'value': [
        f'{chirps_annual.mean():.0f}',
        f'{chirps_annual.idxmin().year} ({chirps_annual.min():.0f} mm)',
        f'{chirps_annual.idxmax().year} ({chirps_annual.max():.0f} mm)',
        f'{pdsi_ts.mean():.3f}',
        f'{drought_frac:.1f}',
        f'{severe_frac:.1f}',
        f'{comp_pct:.1f}',
        f'{r_gw_pdsi:.3f} (p={p_gw_pdsi:.4f})' if r_gw_pdsi is not None else 'N/A',
        f'{r_gw_rain:.3f} (p={p_gw_rain:.4f})' if r_gw_rain is not None else 'N/A',
    ]
}
df_summary = pd.DataFrame(summary)
df_summary.to_csv(TAB_DIR / 'climate_summary_stats.csv', index=False)
print('Saved climate_summary_stats.csv')
print(df_summary.to_string(index=False))

In [ ]:
# ── 8. Final output check ─────────────────────────────────────────────────
expected_outputs = [
    OUT_CLI / 'rainfall_anomaly_monthly.csv',
    OUT_CLI / 'pdsi_annual_mean.tif',
    OUT_CLI / 'drought_frequency.tif',
    OUT_CLI / 'terraclimate_coastal_timeseries.csv',
    FIG_DIR / 'chirps_rainfall_timeseries.png',
    FIG_DIR / 'pdsi_timeseries.png',
    FIG_DIR / 'drought_frequency_map.png',
    FIG_DIR / 'gwsa_vs_pdsi_comparison.png',
    FIG_DIR / 'compound_drought_timeseries.png',
    TAB_DIR / 'drought_episodes_pdsi.csv',
    TAB_DIR / 'climate_summary_stats.csv',
]

print('\n=== Output check ===')
all_ok = True
for p in expected_outputs:
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'  {status}  {p.relative_to(ROOT)}')
    if not p.exists(): all_ok = False

if all_ok:
    print('\nNotebook 08 complete. All outputs present.')
    print('Next: run Notebook 12 (publication maps) or Notebook 11 (validation).')
else:
    print('\nSome outputs missing — check for errors above.')